**Install Libraries**

In [15]:
!pip install -q sentence-transformers faiss-cpu google-genai

**Imports**

In [16]:
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

**Load Embedding Model**

In [17]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


**text_chunks file upload**: *Retrieved from Shubham's Notebook*

In [18]:
from google.colab import files
import json

uploaded = files.upload()  # Select text_chunks.json
json_name = next(name for name in uploaded if name.lower().endswith(".json"))

records = json.loads(uploaded[json_name].decode("utf-8"))
chunks = [record["text"] for record in records]

print(f"Loaded {len(chunks)} chunks")

Saving text_chunks.json to text_chunks (2).json
Loaded 11 chunks


**Generate Embeddings**

In [19]:
embeddings = model.encode(
    chunks,
    convert_to_numpy=True
)

print("Number of chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)

Number of chunks: 11
Embedding shape: (11, 384)


**Create FAISS Index**

In [20]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    embeddings.astype("float32")
)

print("FAISS index created")

FAISS index created


**Dynamic K Function**

In [21]:
def get_dynamic_k(total_chunks):

    k = min(
        max(3, total_chunks // 50),
        10
    )

    return k

**Search Function**

In [22]:
def retrieve_chunks(question):
    query_embedding = model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    k = min(get_dynamic_k(len(chunks)), len(chunks))
    distances, indices = index.search(query_embedding, k)

    results = []
    for distance, idx in zip(distances[0], indices[0]):
        if idx >= 0:
            result = dict(records[int(idx)])
            result["distance"] = float(distance)
            results.append(result)

    return results, k

**Ask Question**

In [23]:
question = input("Ask a question: ")
results, k = retrieve_chunks(question)
print(f"\nTop-{k} relevant chunks:")

Ask a question: What does chunk overlap help preserve?

Top-3 relevant chunks:


**Display Chunks**

In [24]:
for result in results:
    print(
        f"Page {result['page']} | "
        f"Chunk {result['chunk_id']} | "
        f"L2 distance {result['distance']:.3f}"
    )
    print(result["text"])
    print("-" * 60)

Page 5 | Chunk 8 | L2 distance 0.920
label retrieved passages with references such as [page 2, chunk 8] and ask the model to use those labels.
The final answer should be checked against the cited passages before it is presented as reliable.
A simple example
Question: What does chunk overlap help preserve? Retrieved passage: “Chunk overlap repeats a small
amount of text between neighboring passages, helping preserve context when an important sentence
crosses a chunk boundary.” Grounded answer: Chunk overlap helps preserve context when a sentence
spans the boundary between neighboring chunks [page 3].
------------------------------------------------------------
Page 3 | Chunk 4 | L2 distance 0.988
helping preserve context when an important sentence crosses a chunk boundary.
Useful metadata
Each chunk should retain its source document and page number. For example, a chunk record can contain
a chunk identifier, page number, extraction method, and passage text. Keeping these fields alongsid